# Meridian — Return Head Training (Colab)

Trains the `MeridianR` return-prediction head on top of the pre-trained backbone.
Checkpoints save to Google Drive every 5000 steps. Re-run all cells to resume after a session drop.

**Run `colab_fda.ipynb` first** to generate `FDA_SYNTH_*.f32bin` shards — this notebook auto-detects them and mixes them in via `--fda-mix-prob 0.5` / `--fda-loss-weight 0.6` (spec §11.4c).

**Run this before `colab_portfolio_sharpe.ipynb`.**

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
HF_USERNAME   = 'Verdenroz'
HF_TOKEN      = ''            # HF token with read access (required if repos are private)
                              # create at: huggingface.co/settings/tokens
HF_MODEL_REPO = f'{HF_USERNAME}/meridian-checkpoints'
HF_DATA_REPO  = f'{HF_USERNAME}/meridian-data'

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/meridian'
os.makedirs(f'{DRIVE_DIR}/checkpoints/pretrain',  exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints/portfolio', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data/mmaps',            exist_ok=True)
print('Drive ready:', DRIVE_DIR)

In [ ]:
# ── 2. HuggingFace login + install deps ───────────────────────────────────────
import subprocess, sys
subprocess.run([
    'pip', 'install', '-q',
    'numpy>=2.1', 'pandas>=2.2', 'pyyaml>=6.0',
    'hmmlearn>=0.3', 'faiss-cpu>=1.8', 'tokenizers>=0.20',
    'huggingface_hub',
], check=True)

from huggingface_hub import login, snapshot_download, hf_hub_download
if HF_TOKEN:
    login(token=HF_TOKEN)
print('HuggingFace ready.')

In [ ]:
# ── 3. Download training code + symbols file ─────────────────────────────────
import os, shutil

if not os.path.exists('/content/meridian'):
    local = snapshot_download(repo_id=HF_MODEL_REPO, allow_patterns='code/**')
    shutil.copytree(os.path.join(local, 'code'), '/content/meridian')
    print('Code downloaded.')
else:
    print('Code already present.')

# symbols.txt lives at repo root, not inside code/, so fetch it separately.
symbols_path = '/content/meridian/symbols.txt'
if not os.path.exists(symbols_path):
    path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='symbols.txt')
    shutil.copy(path, symbols_path)
    print(f'symbols.txt downloaded → {symbols_path}')
else:
    print('symbols.txt already present.')

In [ ]:
# ── 4. Download data (cached to Drive across sessions) ────────────────────────
# FDA shards are generated by colab_fda.ipynb and mirrored to Drive in that
# notebook's cell 8. On cold start with an empty Drive we snapshot_download
# the real shards from HF; FDA shards only exist on Drive (not uploaded to HF
# because HF caps dirs at 10k files).
#
# Drive FUSE is flaky on dirs with 50k+ files (sporadic EIO on listdir). All
# Drive directory operations here go through a retry wrapper.
import os, shutil, time

def _drive_retry(fn, what, attempts=6, base_sleep=3):
    last = None
    for i in range(attempts):
        try:
            return fn()
        except OSError as e:
            last = e
            print(f'[{what}] Drive EIO (attempt {i + 1}/{attempts}): {e}; sleeping {(i + 1) * base_sleep}s')
            time.sleep((i + 1) * base_sleep)
    raise RuntimeError(
        f'{what} failed after {attempts} retries. Drive FUSE is likely stuck. '
        f'Remount with: from google.colab import drive; drive.flush_and_unmount(); '
        f'drive.mount("/content/drive", force_remount=True); then re-run.'
    ) from last

data_dir  = f'{DRIVE_DIR}/data/mmaps'
train_dir = f'{data_dir}/train'

def _has_data():
    if not os.path.exists(train_dir):
        return False
    # any() over listdir — wrapped so the 50k-file listing retries through EIO.
    entries = _drive_retry(lambda: os.listdir(train_dir), 'list train_dir')
    return any(f.endswith('.f32bin') for f in entries)

has_data = _has_data()

if not has_data:
    snapshot_download(repo_id=HF_DATA_REPO, repo_type='dataset', local_dir=data_dir)
    for prefix in [os.path.join(data_dir, 'data', 'mmaps'), os.path.join(data_dir, 'mmaps')]:
        if os.path.exists(os.path.join(prefix, 'train')):
            for item in os.listdir(prefix):
                src = os.path.join(prefix, item)
                dst = os.path.join(data_dir, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(os.path.join(data_dir, prefix.replace(data_dir + '/', '').split('/')[0]), ignore_errors=True)
            break
    print(f'Data downloaded → {data_dir}')
else:
    print('Data already in Drive.')

In [ ]:
# ── 5. Copy data to local SSD for fast I/O ───────────────────────────────────────────────
# Parallel copy: shutil.copytree is single-threaded, which means each of the
# ~50k Drive-side files waits on its own round-trip. Drive FUSE handles
# concurrent reads well, so a thread pool gets 8–16× speedup. Each file copy
# is retried on EIO because FUSE occasionally chokes on hot directories.
import os, shutil, time, concurrent.futures

local_data = '/content/data/mmaps'
drive_data = f'{DRIVE_DIR}/data/mmaps'

def _copy_one(src, dst, attempts=5, base_sleep=1):
    last = None
    for i in range(attempts):
        try:
            if os.path.exists(dst):
                return  # idempotent — don't re-copy on rerun
            shutil.copy2(src, dst)
            return
        except OSError as e:
            last = e
            time.sleep((i + 1) * base_sleep)
    raise RuntimeError(f'copy failed {src} → {dst}: {last}') from last

def _parallel_copy_dir(src_dir, dst_dir, workers=32):
    os.makedirs(dst_dir, exist_ok=True)
    entries = sorted(os.listdir(src_dir))
    jobs = [(os.path.join(src_dir, f), os.path.join(dst_dir, f)) for f in entries]
    t0 = time.time()
    done = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(_copy_one, s, d) for s, d in jobs]
        for fut in concurrent.futures.as_completed(futures):
            fut.result()
            done += 1
            if done % 2000 == 0 or done == len(jobs):
                rate = done / max(time.time() - t0, 1e-6)
                print(f'    {done}/{len(jobs)} ({rate:.0f} files/s)')
    print(f'  {src_dir} → {dst_dir}: {len(jobs)} files in {time.time() - t0:.1f}s')

if not os.path.exists(f'{local_data}/train'):
    print('Copying data from Drive to local SSD (parallel, one-time per session)...')
    # Copy each split separately — the train/ dir is the slow one (50k files),
    # validation and test are tiny.
    for split in ('train', 'validation', 'test'):
        src_split = os.path.join(drive_data, split)
        if os.path.isdir(src_split):
            _parallel_copy_dir(src_split, os.path.join(local_data, split))
    print('Done.')
else:
    print('Local data already present.')

# Report shard breakdown — portfolio_train.py auto-detects FDA_SYNTH_* shards
# and mixes them in per spec §11.4c. If count is 0, run colab_fda.ipynb first.
train_files = [f for f in os.listdir(f'{local_data}/train') if f.endswith('.f32bin')]
real_shards  = sorted(f for f in train_files if not f.startswith('FDA_SYNTH_'))
synth_shards = sorted(f for f in train_files if f.startswith('FDA_SYNTH_'))
print(f'Real train shards: {len(real_shards)}  |  FDA synth shards: {len(synth_shards)}')
if not synth_shards:
    print('WARNING: no FDA shards detected — training will run real-data-only '
          '(spec §11.4c expects ∼50k synthetic windows). Run colab_fda.ipynb first.')

In [ ]:
# ── 6. Locate pretrain checkpoint (prefers latest step, then best-val, then HF) ─
# Mirrors the pretrain notebook's resume logic so we never silently fall back to
# a stale best-val checkpoint when a newer step save exists.
import os, re, glob, shutil

PRETRAIN_DIR   = f'{DRIVE_DIR}/checkpoints/pretrain'
pretrain_best  = f'{PRETRAIN_DIR}/backbone_pretrain.pt'

def _latest_step_ckpt(ckpt_dir):
    candidates = []
    for path in glob.glob(os.path.join(ckpt_dir, 'backbone_step*.pt')):
        m = re.search(r'backbone_step(\d+)\.pt$', path)
        if m:
            candidates.append((int(m.group(1)), path))
    return max(candidates, key=lambda x: x[0])[1] if candidates else None

latest_step = _latest_step_ckpt(PRETRAIN_DIR)

# Only fetch from HF if Drive has nothing at all.
if latest_step is None and not os.path.exists(pretrain_best):
    try:
        path = hf_hub_download(repo_id=HF_MODEL_REPO, filename='pretrain/backbone_pretrain.pt')
        shutil.copy(path, pretrain_best)
        print(f'Pretrain checkpoint downloaded from HF → {pretrain_best}')
    except Exception as e:
        raise RuntimeError(f'No pretrain checkpoint available locally or on HF: {e}')

if latest_step is not None:
    PRETRAIN_CKPT = latest_step
    print(f'Will resume backbone from latest step checkpoint: {PRETRAIN_CKPT}')
elif os.path.exists(pretrain_best):
    PRETRAIN_CKPT = pretrain_best
    print(f'Will resume backbone from best-val checkpoint: {PRETRAIN_CKPT}')
else:
    raise RuntimeError('No pretrain checkpoint found.')

In [ ]:
# ── 7. Auto-detect GPU ────────────────────────────────────────────────────────
import torch, os
assert torch.cuda.is_available(), 'No GPU — enable it in Runtime → Change runtime type.'

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  ({vram_gb:.0f} GB VRAM)')

# TF32 matmul — free speedup on Ampere+ with negligible numerical impact.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Memory model (spec-faithful, no quality loss):
#   • PTA §11.4b expands B up to 10× (1 orig + 4 perms + 1 sign + 4 vol scales).
#     With --pta-deterministic, all view probs→1.0 so B_eff is constant (10×B),
#     enabling torch.compile with stable shapes. Strictly stronger regularisation.
#   • --pta-micro-batch-size splits the PTA-expanded batch into micro-groups
#     with grad accumulation. Bit-exact identical to one big batch.
#   • bptt_chunk (truncated BPTT) — implementation lever, not spec-pinned.
#   • --gradient-checkpointing auto-ignored when bptt_chunk>0 (chunk T-loop
#     already handles activation rematerialisation).
PTA_DETERMINISTIC = True      # fixed B_eff → compile-friendly, stronger augmentation
DISABLE_REGIME_BALANCE = False
WARMUP_BATCH_SIZE = 8
USE_COMPILE = True            # re-enabled: safe now that PTA shapes are stable
REGIME_POOL_SIZE = 64

if vram_gb >= 75:    # A100 80GB, H100
    BATCH_SIZE    = 4      # with deterministic PTA, B_eff=40 at B=4 (max views)
    BPTT_CHUNK    = 16
    PTA_MICRO_BS  = 10
elif vram_gb >= 35:  # A100 40GB
    BATCH_SIZE    = 4
    BPTT_CHUNK    = 8
    PTA_MICRO_BS  = 4
elif vram_gb >= 22:  # L4 24GB
    BATCH_SIZE    = 4
    BPTT_CHUNK    = 4
    PTA_MICRO_BS  = 2
else:                # T4/V100 16GB
    BATCH_SIZE    = 2
    BPTT_CHUNK    = 2
    PTA_MICRO_BS  = 1
    DISABLE_REGIME_BALANCE = True
    print('WARNING: 16GB tier — dropping to B=2, disabling regime-balanced sampling (spec deviation).')

WORKERS = min(8, os.cpu_count() or 1)
print(
    f'batch={BATCH_SIZE}  warmup_batch={WARMUP_BATCH_SIZE}  bptt_chunk={BPTT_CHUNK}  '
    f'pta_micro_bs={PTA_MICRO_BS}  pta_deterministic={PTA_DETERMINISTIC}  '
    f'regime_pool={REGIME_POOL_SIZE}  compile={USE_COMPILE}  '
    f'regime_balance={not DISABLE_REGIME_BALANCE}  workers={WORKERS}'
)

In [ ]:
# ── 8. Train return head (auto-resumes from Drive) ────────────────────────────
import os, sys, subprocess

# Set DEBUG=True to print per-step [mem-diag] traces (GPU memory peaks per
# micro-group, first 3 steps only). Useful when tuning VRAM headroom.
DEBUG = False

import re, glob

CKPT_PORT      = f'{DRIVE_DIR}/checkpoints/portfolio'
return_best    = f'{CKPT_PORT}/return_head_best.pt'
warmup_ckpt    = f'{CKPT_PORT}/return_cross_asset_warmup.pt'

# Resume priority (mirrors colab_pretrain.ipynb): prefer the most recent step
# checkpoint so training continues from the last periodic save rather than the
# (possibly older) best-val checkpoint.
#   1. return_head_stepN.pt (highest N) — most recent periodic save
#   2. return_head_best.pt — best val monitor
#   3. return_cross_asset_warmup.pt — warmup done, skip to head training
#   4. pretrain backbone — cold start (will run warmup first)
def _latest_step_ckpt(ckpt_dir, head):
    candidates = []
    for path in glob.glob(os.path.join(ckpt_dir, f'{head}_head_step*.pt')):
        m = re.search(rf'{head}_head_step(\d+)\.pt$', path)
        if m:
            candidates.append((int(m.group(1)), path))
    return max(candidates, key=lambda x: x[0])[1] if candidates else None

latest_step = _latest_step_ckpt(CKPT_PORT, 'return')
if latest_step is not None:
    resume_from = latest_step
    print(f'Resuming from latest step checkpoint: {resume_from}')
elif os.path.exists(return_best):
    resume_from = return_best
    print(f'Resuming from best-val checkpoint: {resume_from}')
elif os.path.exists(warmup_ckpt):
    resume_from = warmup_ckpt
    print(f'Resuming from cross-asset warmup: {resume_from}')
else:
    resume_from = PRETRAIN_CKPT
    print(f'Cold start from pretrain backbone: {resume_from}')

cmd = [
    sys.executable, '-u',
    '-m', 'train.portfolio_train',
    '--head',                          'return',
    '--data-dir',                      '/content/data/mmaps',
    '--symbols-file',                  '/content/meridian/symbols.txt',
    '--steps',                         '50000',
    '--batch-size',                    str(BATCH_SIZE),
    '--cross-asset-warmup-batch-size', str(WARMUP_BATCH_SIZE),
    '--seq-len',                       '64',
    '--lr',                            '3e-5',
    '--log-every',                     '10',
    '--eval-every',                    '250',
    '--save-every',                    '250',
    '--bptt-chunk',                    str(BPTT_CHUNK),
    '--pta-micro-batch-size',          str(PTA_MICRO_BS),
    '--regime-pool-size',              str(REGIME_POOL_SIZE),
    '--checkpoint-dir',                CKPT_PORT,
    '--resume-backbone',               resume_from,
    '--workers',                       str(WORKERS),
]
if PTA_DETERMINISTIC:
    cmd.append('--pta-deterministic')
if USE_COMPILE:
    cmd.append('--compile')
if DISABLE_REGIME_BALANCE:
    cmd.append('--disable-regime-balance')
if DEBUG:
    cmd.append('--debug')

env = os.environ.copy()
env['PYTHONPATH']         = '/content/meridian'
env['PYTHONUNBUFFERED']   = '1'
env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

print('Running:', ' '.join(cmd))
print('-' * 80)

process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode != 0:
    raise RuntimeError(f'Training failed with return code {process.returncode}')
print('Training completed.')